In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
dataset = pd.read_csv("IMDB Dataset.csv")

In [3]:
dataset.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df = dataset.copy()

In [5]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [6]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
df.duplicated().sum()

418

In [9]:
df.drop_duplicates(inplace=True)

In [10]:
df.duplicated().sum()

0

In [11]:
# Basic Preprocessing
# Remove HTML tags
# Lowercase
# Remove Stopwords

In [12]:
import re
def remove_tags(raw_text):
    return re.sub(re.compile('<.*?>'), '', raw_text)

In [13]:
df['review'] = df['review'].apply(remove_tags)

In [14]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [15]:
df['review'] = df['review'].apply(lambda x: x.lower())

In [16]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list])\
    .apply(lambda x: " ".join(x))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Narex\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
X = df['review']
y = df['sentiment']

In [18]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [19]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
X_train.shape

(39665,)

In [21]:
# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000)

In [22]:
X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

In [23]:
X_train_bow.shape

(39665, 10000)

In [24]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(X_train_bow, y_train)

GaussianNB()

In [25]:
y_pred = gnb.predict(X_test_bow)

from sklearn.metrics import accuracy_score, classification_report
accuracy_score(y_test, y_pred)

0.7292527982252698

In [26]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.86      0.76      4939
           1       0.82      0.59      0.69      4978

    accuracy                           0.73      9917
   macro avg       0.75      0.73      0.72      9917
weighted avg       0.75      0.73      0.72      9917



In [27]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train_bow, y_train)

y_pred = rf.predict(X_test_bow)
accuracy_score(y_test, y_pred)

0.844711102147827

In [28]:
# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=3000, ngram_range=(1,2))

X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

rf = RandomForestClassifier()
rf.fit(X_train_bow, y_train)

y_pred = rf.predict(X_test_bow)
accuracy_score(y_test, y_pred)

0.835232429162045

### TFIDF

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [30]:
tfidf = TfidfVectorizer()

X_train_tfidf = cv.fit_transform(X_train).toarray()
X_test_tfidf = cv.transform(X_test).toarray()

rf = RandomForestClassifier()
rf.fit(X_train_tfidf, y_train)

y_pred = rf.predict(X_test_tfidf)
accuracy_score(y_test, y_pred)

0.8392659070283351

### Word2Vec

In [32]:
import gensim

In [33]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [34]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [35]:
model = gensim.models.Word2Vec(
        window=10,
     min_count=2
)

In [36]:
model.build_vocab(story)

In [37]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(29539096, 30891345)

In [38]:
len(model.wv.index_to_key)

61843

In [39]:
def document_vector(doc):
    # remove oov words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [41]:
document_vector(df['review'].values[5])

array([ 0.11259992,  0.1298361 , -0.38947326, -0.45973757,  0.38623092,
       -0.09921208,  0.13063991,  0.34890515,  0.05564687,  0.27360773,
        0.06318825, -0.443012  ,  0.545482  ,  0.01778973,  0.62571114,
       -0.00311202, -0.24889386,  0.33595273, -0.00152836, -0.4798888 ,
        0.1881134 , -0.41933042,  0.07088304,  0.15269291, -0.4810021 ,
       -0.630359  , -0.04248019,  0.33742332, -0.68969476, -0.09335663,
        0.49913505,  0.02889879, -0.20817453, -0.43930992,  0.35836342,
        0.00485918,  0.2133833 , -0.19844325, -0.35965627, -0.2517829 ,
       -0.39422747,  0.13894096, -0.03518913, -0.26354983,  0.29472944,
        0.04971781, -0.23878165,  0.41535014, -0.11170345,  0.3340601 ,
        0.25550807,  0.07595494,  0.12563682, -0.01150738, -0.17466885,
        0.4587218 , -0.20556389, -0.1248244 ,  0.03265646,  0.2691713 ,
        0.11314781,  0.34623337,  0.44930044, -0.29790992, -0.8966813 ,
        0.6350821 , -0.26909405,  0.01800664, -0.35147145,  0.33

In [42]:
from tqdm import tqdm

In [ ]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

In [ ]:
X = np.array(X)

In [ ]:
X.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(df['sentiment'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
accuracy_score(y_test, y_pred)

In [ ]:
# Pretrained Word2Vec